In [25]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jorda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jorda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jorda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\jorda\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jorda\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [26]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
import pickle
from nltk.classify import NaiveBayesClassifier

In [27]:
# Load Dataset
import kagglehub
from pathlib import Path

# Massive Rotten Tomatoes Movies and Reviews, downloaded once to the kagglehub cache
RT_DIR = Path(kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews"))
movies = pd.read_csv(RT_DIR / "rotten_tomatoes_movies.csv", usecols=["id", "title", "audienceScore"]).dropna()
reviews = pd.read_csv(RT_DIR / "rotten_tomatoes_movie_reviews.csv", usecols=["id", "reviewText"]).dropna()

# Same layout as before: Title, Rating (0-10, from the audience score) and Review
dataset = (
    reviews.merge(movies, on="id")
    .assign(Rating=lambda d: d["audienceScore"] / 10)
    .rename(columns={"title": "Title", "reviewText": "Review"})[["Title", "Rating", "Review"]]
    .sample(10000, random_state=42)
    .reset_index(drop=True)
)
dataset = dataset.dropna().head(1000)

dataset.head()

,Title,Rating,Review
0,The Idea of You,6.4,"This film, as well as the reaction to it, is a..."
1,Kingdom of the Planet of the Apes,7.3,"I'm a big fan of all the planet of the apes, a..."
2,Unfrosted,5.5,Pretty much the worst criticism you can lay on...
3,The Fall Guy,7.3,Just got out of the Austin premier at SXSW and...
4,Challengers,7.7,This is a tough one. I liked the concept and t...


In [28]:
# Preprocess Data
eng_stopwords = stopwords.words('english')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = word_tokenize(sentence)
    sentence = [word for word in sentence if word not in eng_stopwords]
    sentence = [word for word in sentence if word.isalpha()]
    sentence = [stemmer.stem(word) for word in sentence]
    sentence = [lemmatizer.lemmatize(word) for word in sentence]

    return sentence

In [29]:
# Extract Features
def extract_features(review):
    review = preprocess_sentence(review)
    return {word: True for word in review if word in review} # {'kata': True}

In [30]:
def train_model():

    # Extract Data
    X = dataset['Review']
    y = ['Positive' if rating > 5 else 'Negative' for rating in dataset['Rating']]

    all_reviews = ''.join(X)
    all_tokens = preprocess_sentence(all_reviews)
    feature_sets = [(extract_features(review), rating) for review, rating in zip(X, y)]

    # Split Data
    train_count = int(len(feature_sets) * 0.8)
    train_set = feature_sets[:train_count]
    test_set = feature_sets[train_count:]

    # Classify
    global classifier
    classifier = NaiveBayesClassifier.train(train_set)

In [31]:
def predict_review(review):
    review = extract_features(review)
    return classifier.classify(review)

In [32]:
vectorizer = TfidfVectorizer()

def get_similarity_count(query, target):
    target_v = vectorizer.fit_transform(target)
    query_v = vectorizer.transform([query])
    return cosine_similarity(query_v, target_v)

def get_top_movies(query):
    similarity = get_similarity_count(query, dataset['Review'])

    data = {'title': dataset['Title'], 'review': dataset['Review'], 'similarity': similarity[0]}

    df = pd.DataFrame(data).sort_values(by='similarity', ascending=False).head(2)
    # print(df)
    print(f"1: {df['title'].mode()[0]}")
    print(f"2: {df['title'].mode()[1]}")

In [33]:
nlp = spacy.load('en_core_web_sm')

def get_entities(review):
    doc = nlp(review)
    categories = {}

    for ent in doc.ents:
        label = ent.label_
        # if label not in ["LANGUAGE", "LOC"]: continue
        if label not in categories.keys():
            categories[label] = []

        categories[label].append(ent.text)

    return categories

In [34]:
def check_model():
    try:
        with open('model.pkl', 'rb') as f:
            global classifier
            classifier = pickle.load(f)
            print("Model loaded")
    except:
        with open('model.pkl', 'wb') as f:
            print("Training model")
            train_model()
            pickle.dump(classifier, f)

check_model()

Model loaded


In [35]:
def main():
    check_model()

    review = ""
    while True:

        print("MOVIE RECOMMENDATION APPLICATION BASED ON REVIEWS")
        if review == "":
          print("Your Review: NO REVIEW")
          print("Your Review Category: UNKNOWN")
        else:
          print(f"Your Review: {review}")
          print(f"Your Review Category: {predict_review(review)}")

        print("1. Write Your Review ")
        print("2. View Movie Recommendation")
        print("3. View Named Entity Recognition")
        print("4. Exit")

        choice = input(">> ")

        if choice == '1':
          while True:
            review = input("Input your review: ")
            if len(review.split()) >= 20:
              print(predict_review(review))
              print()
              print("[>] Press enter to continue ...")
              break
            else:
              print("Has to be at least 20 words...")

        elif choice == '2':
          if review == '': continue
          print("TOP 2 MOVIE RECOMMENDATION FOR YOU:")
          get_top_movies(review)
          print()
          print("[>] Press enter to continue ...")

        elif choice == '3':
          if review == "" : continue
          entities = get_entities(review)
          print("CATEGORIZED NAMED ENTITIES:")
          for entity in entities:
              print(f"{entity}: {' '.join(entities[entity])}")
          print()
          print("[>] Press enter to continue ...")

        elif choice == '4':
          return

        else:
          print("Insert a valid option...")

        input()

In [36]:
main()

Model loaded
MOVIE RECOMMENDATION APPLICATION BASED ON REVIEWS
Your Review: NO REVIEW
Your Review Category: UNKNOWN
1. Write Your Review 
2. View Movie Recommendation
3. View Named Entity Recognition
4. Exit
